# Train a World Model on Your Own Environment

This notebook walks through the full WorldKit workflow: record data from a
Gymnasium environment, train a world model, make predictions, and plan actions.

We'll use CartPole as our example — but the same steps work for any Gymnasium
environment that renders RGB frames.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DilpreetBansi/worldkit/blob/main/notebooks/02_train_custom_env.ipynb)

**Time to run:** ~5 minutes on Colab T4

In [ ]:
!pip install "worldkit[envs,train]" -q

## 1. Explore the Environment

First, let's look at what CartPole gives us. WorldKit has a built-in
environment registry that knows about common Gymnasium environments.

In [ ]:
from worldkit.envs.registry import registry

# Search for CartPole
results = registry.search("cart")
for env in results:
    print(f"{env.env_id}")
    print(f"  Gym ID:      {env.gym_id}")
    print(f"  Action dim:  {env.action_dim} ({env.action_type})")
    print(f"  Obs shape:   {env.observation_shape}")
    print(f"  Description: {env.description}")

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

# Create the environment and render a frame
env = gym.make("CartPole-v1", render_mode="rgb_array")
obs, info = env.reset(seed=42)
frame = env.render()

print(f"Frame shape: {frame.shape}")
print(f"Frame dtype: {frame.dtype}")
print(f"Action space: {env.action_space}")

plt.figure(figsize=(4, 3))
plt.imshow(frame)
plt.title("CartPole initial state")
plt.axis("off")
plt.tight_layout()
plt.show()

env.close()

## 2. Record Training Data

The `Recorder` class wraps a Gymnasium environment and collects episodes
into an HDF5 file — the format WorldKit expects for training.

We'll record 200 episodes with a random policy. For better models,
you could use a trained policy or human demonstrations.

In [ ]:
from worldkit.data import Recorder

env = gym.make("CartPole-v1", render_mode="rgb_array")
rec = Recorder(env, output="cartpole_data.h5")

# Record 200 episodes with random actions
data_path = rec.record(episodes=200, policy="random", max_steps_per_episode=200)

print(f"Data saved to: {data_path}")

In [ ]:
import h5py

# Inspect the recorded dataset
with h5py.File("cartpole_data.h5", "r") as f:
    print("Keys:", list(f.keys()))
    for key in f.keys():
        print(f"  {key}: shape={f[key].shape}, dtype={f[key].dtype}")

In [ ]:
# Visualize a few frames from the dataset
with h5py.File("cartpole_data.h5", "r") as f:
    pixels = f["pixels"]
    fig, axes = plt.subplots(1, 5, figsize=(15, 3))
    for i, ax in enumerate(axes):
        # Show frames from episode 0 at different timesteps
        t = i * (pixels.shape[1] // 5)
        ax.imshow(pixels[0, t])
        ax.set_title(f"t={t}")
        ax.axis("off")
    plt.suptitle("Episode 0 — sample frames", fontsize=14)
    plt.tight_layout()
    plt.show()

## 3. Train the World Model

We'll use the `nano` config (5M parameters) which trains quickly on Colab.
CartPole has a discrete action space with 2 actions (left/right), so `action_dim=1`.

In [ ]:
from worldkit import WorldModel

model = WorldModel.train(
    data="cartpole_data.h5",
    config="nano",
    action_dim=1,
    epochs=20,
    batch_size=32,
    device="auto",  # Uses GPU if available
)

print(f"\nParameters: {model.num_params:,}")
print(f"Latent dim: {model.latent_dim}")
print(f"Device:     {model.device}")

In [ ]:
# Save the trained model
model.save("cartpole_nano.wk")
print("Model saved: cartpole_nano.wk")

## 4. Make Predictions

Now let's see what our model has learned. We'll encode observations
and predict future states given actions.

In [ ]:
# Get a fresh frame from the environment
env = gym.make("CartPole-v1", render_mode="rgb_array")
obs, _ = env.reset(seed=0)
frame = env.render()
env.close()

# Encode it
z = model.encode(frame)
print(f"Encoded frame to latent: {z.shape}")

# Predict 10 steps of pushing right (action=1)
push_right = [(1.0,)] * 10
result_right = model.predict(frame, actions=push_right)

# Predict 10 steps of pushing left (action=0)
push_left = [(0.0,)] * 10
result_left = model.predict(frame, actions=push_left)

print(f"\nRight trajectory: {result_right.latent_trajectory.shape}, confidence={result_right.confidence:.4f}")
print(f"Left trajectory:  {result_left.latent_trajectory.shape}, confidence={result_left.confidence:.4f}")

In [ ]:
# Compare how the latent state diverges for different actions
norms_right = [np.linalg.norm(result_right.latent_trajectory[i]) for i in range(result_right.steps)]
norms_left = [np.linalg.norm(result_left.latent_trajectory[i]) for i in range(result_left.steps)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot latent norms
axes[0].plot(range(1, len(norms_right) + 1), norms_right, "o-", color="steelblue", label="Push right")
axes[0].plot(range(1, len(norms_left) + 1), norms_left, "s-", color="coral", label="Push left")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Latent norm")
axes[0].set_title("Latent norm evolution")
axes[0].legend()

# Plot L2 distance between the two trajectories
min_steps = min(result_right.steps, result_left.steps)
distances = [
    np.linalg.norm(result_right.latent_trajectory[i] - result_left.latent_trajectory[i])
    for i in range(min_steps)
]
axes[1].plot(range(1, min_steps + 1), distances, "D-", color="mediumpurple")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("L2 distance")
axes[1].set_title("Latent divergence: left vs right")

plt.tight_layout()
plt.show()

## 5. Plan Actions

The CEM planner can find action sequences that move the system
from a current state toward a goal state — without any reward function.

Let's plan how to get from an upright pole to a different configuration.

In [ ]:
# Create start and goal observations
env = gym.make("CartPole-v1", render_mode="rgb_array")

# Start state: pole upright
env.reset(seed=10)
start_frame = env.render()

# Goal state: run the env forward to get a tilted pole
env.reset(seed=99)
for _ in range(5):
    env.step(1)  # Push right a few times
goal_frame = env.render()
env.close()

# Visualize start and goal
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(start_frame)
axes[0].set_title("Start")
axes[0].axis("off")
axes[1].imshow(goal_frame)
axes[1].set_title("Goal")
axes[1].axis("off")
plt.tight_layout()
plt.show()

# Plan
plan = model.plan(
    current_state=start_frame,
    goal_state=goal_frame,
    max_steps=15,
    n_candidates=200,
    n_elite=20,
    n_iterations=5,
)

print(f"Planned {len(plan.actions)} actions")
print(f"Expected cost:       {plan.expected_cost:.4f}")
print(f"Success probability: {plan.success_probability:.4f}")
print(f"Planning time:       {plan.planning_time_ms:.1f} ms")

In [ ]:
# Visualize the planned actions
actions_arr = np.array(plan.actions).flatten()

fig, ax = plt.subplots(figsize=(8, 3))
ax.step(range(len(actions_arr)), actions_arr, where="mid", color="steelblue", linewidth=2)
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5, label="Decision boundary")
ax.set_xlabel("Step")
ax.set_ylabel("Action value")
ax.set_title("Planned action sequence (0=left, 1=right)")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Evaluate with Plausibility Scoring

Let's verify the model has learned something about CartPole's physics.
Real CartPole trajectories should score higher than random frames.

In [ ]:
# Collect a real trajectory
env = gym.make("CartPole-v1", render_mode="rgb_array")
env.reset(seed=42)
real_frames = []
for _ in range(8):
    action = env.action_space.sample()
    env.step(action)
    real_frames.append(env.render())
env.close()

# Create a random sequence
random_frames = [np.random.randint(0, 255, real_frames[0].shape, dtype=np.uint8) for _ in range(8)]

# Score both
score_real = model.plausibility(real_frames)
score_random = model.plausibility(random_frames)

print(f"Real CartPole trajectory: {score_real:.4f}")
print(f"Random frames:           {score_random:.4f}")

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["Real trajectory", "Random frames"], [score_real, score_random],
       color=["steelblue", "coral"])
ax.set_ylabel("Plausibility score")
ax.set_title("Has the model learned CartPole physics?")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## Bonus: Use a Custom Policy for Better Data

Random policies produce short, low-quality episodes in CartPole.
You can pass a custom policy function to the `Recorder` for richer data.

In [ ]:
# Simple heuristic policy: push in the direction the pole is leaning
def balance_policy(obs):
    """Push right if pole tilts right, left if it tilts left."""
    pole_angle = obs[2]  # CartPole obs[2] is the pole angle
    return 1 if pole_angle > 0 else 0

env = gym.make("CartPole-v1", render_mode="rgb_array")
rec = Recorder(env, output="cartpole_balanced.h5")
path = rec.record(episodes=50, policy=balance_policy, max_steps_per_episode=500)
print(f"Balanced data saved to: {path}")

# Check episode lengths — should be much longer than random
with h5py.File("cartpole_balanced.h5", "r") as f:
    print(f"Dataset shape: {f['pixels'].shape}")

## Next Steps

You've trained a world model on a real Gymnasium environment. Try:

- **Different environments**: Replace `CartPole-v1` with `Pendulum-v1`, `Acrobot-v1`, or any env with `render_mode="rgb_array"`
- **Larger models**: Switch `config="nano"` to `"base"` or `"large"` for better predictions
- **More epochs**: Train for 50-100 epochs for higher quality
- **Better policies**: Use trained RL agents or human demos instead of random

| Notebook | What's next |
|----------|-------------|
| [03 — Anomaly Detection](03_anomaly_detection.ipynb) | Use plausibility scoring for anomaly detection |
| [04 — Latent Probing](04_latent_probing.ipynb) | Investigate what the model learned |
| [05 — Model Comparison](05_model_comparison.ipynb) | Compare different model sizes |
| [06 — Export & Deploy](06_export_deploy.ipynb) | Export your model to ONNX |
| [07 — Plan Robot Actions](07_plan_robot_actions.ipynb) | Advanced planning techniques |